# CERF check

Verifying that the CERF API matches the Excel export

In [1]:
%load_ext jupyter_black
%load_ext autoreload
%autoreload 2

In [2]:
from io import BytesIO

import ocha_stratus as stratus
import pandas as pd

from src.datasources import cerf

In [3]:
blob_name = "cerf/AllocationsByYear.xlsx"

In [4]:
df_cerf_excel = pd.read_excel(
    BytesIO(stratus.load_blob_data(blob_name, container_name="global"))
)

In [5]:
test_country_name = "Afghanistan"

In [21]:
df_cerf_excel.dtypes

Country                       str
Allocation date    datetime64[us]
Amount in US$               int64
Window                        str
Emergency                     str
Status                        str
dtype: object

In [19]:
df_cerf_excel_test = df_cerf_excel[
    df_cerf_excel["Country"] == test_country_name
].sort_values("Allocation date", ascending=False)

In [9]:
df_cerf_api = cerf.load_cerf_applications()

In [94]:
df_cerf_api.dtypes

TableName                                        str
AllocationStatus                                 str
ApplicationID                                    str
Year                                             str
ApplicationCode                                  str
CountryID                                        str
CountryName                                      str
CountryCode                                      str
WindowID                                         str
WindowFullName                                   str
EmergencyTypeID                                  str
EmergencyTypeName                                str
EmergencyGroupForGlobalReporting                 str
ApplicationTitle                                 str
EmergencyGroupID                                 str
EmergencyGroupName                               str
EmergencyCategoryID                              str
EmergencyCategoryName                            str
RegionID                                      

In [95]:
df_cerf_api["CN_ERC_EndorsementDate"] = pd.to_datetime(
    df_cerf_api["CN_ERC_EndorsementDate"]
)

In [96]:
df_cerf_api[["TotalAmountApproved", "CN_AmountRequested"]] = (
    df_cerf_api[["TotalAmountApproved", "CN_AmountRequested"]].astype(float).astype(int)
)

In [66]:
cols = ["CN_ERC_EndorsementDate", "TotalAmountApproved"]
df_cerf_api_test = df_cerf_api[
    df_cerf_api["CountryName"] == test_country_name
].sort_values("CN_ERC_EndorsementDate", ascending=False)

In [67]:
len(df_cerf_excel)

1588

In [68]:
len(df_cerf_api)

1588

In [71]:
df_merge = df_cerf_excel.merge(
    df_cerf_api,
    left_on=["Country", "Amount in US$", "Allocation date"],
    right_on=["CountryName", "TotalAmountApproved", "CN_ERC_EndorsementDate"],
    how="outer",
)

In [103]:
column_mapping = {
    old: new
    for old, new in zip(
        ["CountryName", "TotalAmountApproved", "CN_ERC_EndorsementDate"],
        ["Country", "Amount in US$", "Allocation date"],
        strict=True,
    )
}

In [104]:
column_mapping

{'CountryName': 'Country', 'TotalAmountApproved': 'Amount in US$', 'CN_ERC_EndorsementDate': 'Allocation date'}